In [76]:
import xarray as xr

pdataset = xr.open_dataset(r"G:\Shared drives\Sutter-Fella Lab\ALS_Beamtimes\2026\Yi-Ru_Feb2026\processed\giwaxs_combined.nc")
pdataset.experiment.values

array(['ClMBAI_0M5_1 004910 Images', 'ClMBAI_120c_aging_1 004871 Images',
       'ClMBAI_aging_from_100C_2min_anneal 004915 Images',
       'ClMBAI_anneal_2min_120c_aging_1 004873 Images',
       'MBAI_0M5_3 004909 Images',
       'MBAI_aging_from_100C_2min_anneal 004913 Images',
       'MeOMBAI_0M5_2 004912 Images',
       'MeOMBAI_120c_aging_1 004868 Images',
       'MeOMBAI_aging_from_100C_2min_anneal 004914 Images',
       'MeOMBAI_anneal_2min_120c_aging_1 004872 Images'], dtype='<U49')

In [ ]:
import json
from pprint import pprint
from pathlib import Path

dirs = [Path(json.loads(v)['source_dir']) for v in pdataset.attrs.values()]

pprint(dirs)

[WindowsPath('Y:/2026/schwartz_Feb2026/schwartz_2_Feb2026/YiRu_feb2026_2/260220/ClMBAI_0M5_1 004910 Images'),
 WindowsPath('Y:/2026/schwartz_Feb2026/schwartz_2_Feb2026/YiRu_feb2026/in-situ/260214/ClMBAI_120c_aging_1 004871 Images'),
 WindowsPath('Y:/2026/schwartz_Feb2026/schwartz_2_Feb2026/YiRu_feb2026_2/260220/ClMBAI_aging_from_100C_2min_anneal 004915 Images'),
 WindowsPath('Y:/2026/schwartz_Feb2026/schwartz_2_Feb2026/YiRu_feb2026/in-situ/260214/ClMBAI_anneal_2min_120c_aging_1 004873 Images'),
 WindowsPath('Y:/2026/schwartz_Feb2026/schwartz_2_Feb2026/YiRu_feb2026_2/260220/MBAI_0M5_3 004909 Images'),
 WindowsPath('Y:/2026/schwartz_Feb2026/schwartz_2_Feb2026/YiRu_feb2026_2/260220/MBAI_aging_from_100C_2min_anneal 004913 Images'),
 WindowsPath('Y:/2026/schwartz_Feb2026/schwartz_2_Feb2026/YiRu_feb2026_2/260220/MeOMBAI_0M5_2 004912 Images'),
 WindowsPath('Y:/2026/schwartz_Feb2026/schwartz_2_Feb2026/YiRu_feb2026/in-situ/260214/MeOMBAI_120c_aging_1 004868 Images'),
 WindowsPath('Y:/2026/schwa

In [80]:
Path("attempt_2", *(dirs[0].parts[4:]), dirs[0].stem.replace(" Images", ".h5"))

WindowsPath('attempt_2/YiRu_feb2026_2/260220/ClMBAI_0M5_1 004910 Images/ClMBAI_0M5_1 004910.h5')

In [81]:
[Path(base_dir, "attempt_2", *(dir.parts[4:-1]), dir.stem.replace(" Images", ".h5")) for dir in dirs]
# G:\Shared drives\Sutter-Fella Lab\ALS_Beamtimes\2026\Yi-Ru_Feb2026\YiRu_feb2026_2\260220\ClMBAI_0M5_1 004910.h5

[WindowsPath('G:/Shared drives/Sutter-Fella Lab/ALS_Beamtimes/2026/Yi-Ru_Feb2026/attempt_2/YiRu_feb2026_2/260220/ClMBAI_0M5_1 004910.h5'),
 WindowsPath('G:/Shared drives/Sutter-Fella Lab/ALS_Beamtimes/2026/Yi-Ru_Feb2026/attempt_2/YiRu_feb2026/in-situ/260214/ClMBAI_120c_aging_1 004871.h5'),
 WindowsPath('G:/Shared drives/Sutter-Fella Lab/ALS_Beamtimes/2026/Yi-Ru_Feb2026/attempt_2/YiRu_feb2026_2/260220/ClMBAI_aging_from_100C_2min_anneal 004915.h5'),
 WindowsPath('G:/Shared drives/Sutter-Fella Lab/ALS_Beamtimes/2026/Yi-Ru_Feb2026/attempt_2/YiRu_feb2026/in-situ/260214/ClMBAI_anneal_2min_120c_aging_1 004873.h5'),
 WindowsPath('G:/Shared drives/Sutter-Fella Lab/ALS_Beamtimes/2026/Yi-Ru_Feb2026/attempt_2/YiRu_feb2026_2/260220/MBAI_0M5_3 004909.h5'),
 WindowsPath('G:/Shared drives/Sutter-Fella Lab/ALS_Beamtimes/2026/Yi-Ru_Feb2026/attempt_2/YiRu_feb2026_2/260220/MBAI_aging_from_100C_2min_anneal 004913.h5'),
 WindowsPath('G:/Shared drives/Sutter-Fella Lab/ALS_Beamtimes/2026/Yi-Ru_Feb2026/attempt

In [ ]:
import fabio
import matplotlib.pyplot as plt
import h5py
import matplotlib.colors as colors
import numpy as np


base_dir = Path(r'G:\Shared drives\Sutter-Fella Lab\ALS_Beamtimes\2026\Yi-Ru_Feb2026')

h5_files = [Path(base_dir, "attempt_2", *(dir.parts[4:-1]), dir.stem.replace(" Images", ".h5")) for dir in dirs]

target_frame_indices = [0, -1]

for h5_file in h5_files[-1:]:
    with h5py.File(h5_file, 'r') as f:
        imgs = list(f['Data/images'].keys())

        for idx in target_frame_indices:

            img_name = imgs[idx]

            img = f[f'Data/images/{img_name}'][:]

            img_masked = np.ma.masked_where(img >= 2**32-1, img)
            fig, ax = plt.subplots(figsize=(7, 7), dpi=300, layout='constrained')

            img_masked = np.ma.filled(img_masked, 0)
            ax.set_facecolor("k")
            im = ax.imshow(img_masked, cmap='gray', norm=colors.LogNorm(vmin=16780000, vmax=np.nanpercentile(img_masked, 95)))
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(f"{h5_file.stem} - {img_name}")
            fig.savefig(base_dir/ "processed/single_frames" / f"{h5_file.stem}_{img_name}.png")
            plt.close(fig)
